# VisionCoder OpenEnv — RL Training with Full-Episode GRPO

**Scaler × Meta PyTorch Hackathon 2026**

This notebook trains `Qwen/Qwen3.5-2B` (a unified vision+text model) to generate HTML from screenshots using reinforcement learning.

The environment renders every HTML submission in a real headless browser (Playwright), computes an 8-component visual reward, and uses Full-Episode GRPO to propagate terminal rewards back through the entire episode trajectory.

**Required runtime:** GPU — A100 40GB or larger recommended. Go to **Runtime → Change runtime type → A100**.

---

## Architecture

```
Reference screenshot ──► Developer (Qwen3.5-2B) ──► HTML ──► /step ──► reward
                                  ▲                              │
                         CSS fix instructions ◄── Critic ◄───────┘
```

**Training algorithm (Full-Episode GRPO):**
- Sample K=4 complete trajectories per task
- Score each trajectory: `R_terminal + 0.2 × Σ(improvement deltas)`
- Compute group-relative advantages, apply LoRA gradient update


## Step 1 — Install Dependencies

Installs the VisionCoder OpenEnv package, Playwright for headless browser rendering, and ML dependencies.

In [ ]:
# Clone the repo
!git clone https://github.com/amaljoe/vision-coder-openenv.git
%cd vision-coder-openenv

# Install the package and dependencies
!pip install -e . --quiet
!pip install peft transformers accelerate torch torchvision --quiet

# Install Playwright (headless Chromium for browser rendering)
!pip install playwright --quiet
!playwright install chromium --with-deps

## Step 2 — Set API Token

Set your HuggingFace token to download `Qwen/Qwen3.5-2B`. You can create a read token at https://huggingface.co/settings/tokens

In [ ]:
import os

# Set your HuggingFace token (required for model download)
os.environ["HF_TOKEN"] = "hf_your_token_here"  # replace with your token

# Training config
os.environ["TRAIN_MODEL"] = "Qwen/Qwen3.5-2B"   # base model
os.environ["CHECKPOINT_DIR"] = "checkpoints"     # where to save LoRA weights

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND — switch to A100 runtime'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## Step 3 — Run RL Training

Trains the Developer agent for 20 episodes with K=4 rollouts each.

The training script automatically:
1. Starts the environment server (FastAPI + Playwright)
2. Loads `Qwen3.5-2B` with LoRA (rank=16, 0.49% trainable parameters)
3. Runs Full-Episode GRPO: sample trajectories → compute advantages → update LoRA

**Expected runtime:** ~2 hours on A100 for 20 episodes. Reduce `--episodes` for a shorter demo run.

In [ ]:
# Run training — 5 episodes demo (set --episodes 20 for full training run)
!python train.py \
    --phase developer \
    --episodes 5 \
    --k-rollouts 4 \
    --model Qwen/Qwen3.5-2B \
    --checkpoint-dir checkpoints/colab_run

## Step 4 — Plot Training Rewards

Load the per-episode reward log and visualize training progress.

In [ ]:
import csv
import matplotlib.pyplot as plt
from collections import defaultdict

# Load reward log
rewards_by_difficulty = defaultdict(list)
with open("checkpoints/colab_run/reward_log.csv") as f:
    for row in csv.DictReader(f):
        rewards_by_difficulty[row["difficulty"]].append(float(row["mean_reward"]))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
colors = {"easy": "#3b82f6", "medium": "#22c55e", "hard": "#ef4444"}
for difficulty, rewards in sorted(rewards_by_difficulty.items()):
    episodes = list(range(1, len(rewards) + 1))
    ax.plot(episodes, rewards, marker="o", label=difficulty, color=colors.get(difficulty))

ax.set_xlabel("Episode")
ax.set_ylabel("Mean Reward (0-1)")
ax.set_title("VisionCoder GRPO Training — Developer Agent")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_curve_colab.png", dpi=150)
plt.show()
print("Saved: training_curve_colab.png")

## Step 5 — Run Inference with Trained Model

Compare the trained LoRA against the base model.

In [ ]:
# Evaluate trained vs base model using the bundled eval script
!python eval_lora.py \
    --lora-path checkpoints/colab_run/developer_final \
    --model Qwen/Qwen3.5-2B \
    --episodes 2

## Step 6 — Run Inference Against the Live HF Space

You can also run inference against the live environment hosted on HuggingFace Spaces — no local server needed.

In [ ]:
import os

# Use HF Serverless Inference API (set your token above)
os.environ["API_BASE_URL"] = "https://router.huggingface.co/v1"
os.environ["MODEL_NAME"] = "Qwen/Qwen3.5-35B-A3B"   # larger model via HF router
os.environ["MAX_STEPS"] = "3"
os.environ["INFERENCE_SERVER_PORT"] = "18080"

# Start local env server and run inference
!python inference.py

## Links

- **HF Space (live environment):** https://huggingface.co/spaces/amaljoe88/vision-coder-openenv  
- **Blog:** https://huggingface.co/spaces/amaljoe88/vision-coder-openenv/blob/main/blog.md  
- **Interactive demo:** https://amaljoe.github.io/vision-coder-openenv/  
- **GitHub:** https://github.com/amaljoe/vision-coder-openenv  
